# 02 - Fetch Players (Rosters)

**Database: MySQL** (same database as notebook 01: `NHL_Analytics_Hub`)

**Endpoint:** `GET https://api-web.nhle.com/v1/roster/{team_abbrev}/current`

Run notebook 01 first: this notebook reads the `teams` table it created.

In [1]:
%pip install requests pandas mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import requests
import pandas as pd
import mysql.connector
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

ROSTER_URL = "https://api-web.nhle.com/v1/roster/{team_abbrev}/current"
REQUEST_DELAY = 0.5

# Same settings as notebook 01
DB_HOST     = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT     = int(os.getenv("DB_PORT", "3306"))
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "root")   # <-- change to YOUR MySQL password
DB_NAME     = os.getenv("DB_NAME", "NHL_Analytics_Hub")

def get_connection():
    return mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD,
        database=DB_NAME, connection_timeout=15,
    )

## Load the team list built in notebook 01

In [3]:
conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev")
teams_df = pd.DataFrame(cur.fetchall(), columns=["team_id", "team_abbrev"])
if teams_df.empty:
    raise RuntimeError("teams table is empty - run notebook 01 first.")

team_id_lookup = dict(zip(teams_df.team_abbrev, teams_df.team_id.astype(int)))
print(f"{len(teams_df)} teams loaded")
display(teams_df.head())

32 teams loaded


,team_id,team_abbrev
0,26,ANA
1,4,BOS
2,18,BUF
3,1,CAR
4,28,CBJ


## Create the `players` table

The original notebook never created this table, so the insert would fail with "no such table".

In [4]:
PLAYERS_SQL = """
CREATE TABLE IF NOT EXISTS players (
    player_id BIGINT PRIMARY KEY,
    team_id INT NOT NULL,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    position VARCHAR(10),
    jersey_number INT,
    birth_date DATE,
    birth_country VARCHAR(10),
    height_cm FLOAT,
    weight_kg FLOAT,
    shoots_catches VARCHAR(5),
    headshot_url TEXT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""
cur.execute(PLAYERS_SQL)
conn.commit()
print("players table ready")

players table ready


## HTTP helper (timeout + automatic retries)

In [5]:
session = requests.Session()
retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"], respect_retry_after_header=True)
session.mount("https://", HTTPAdapter(max_retries=retry))

def fetch_roster(abbrev):
    resp = session.get(ROSTER_URL.format(team_abbrev=abbrev), timeout=(10, 30))
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, dict):
        raise ValueError("Unexpected roster response")
    return data

## Inspect one roster response before writing the parser

In [6]:
sample_abbrev = teams_df.team_abbrev.iloc[0]
sample_data = fetch_roster(sample_abbrev)

print("Top-level keys:", list(sample_data.keys()))
print("Forwards:", len(sample_data.get("forwards", [])))
print("Defensemen:", len(sample_data.get("defensemen", [])))
print("Goalies:", len(sample_data.get("goalies", [])))
display(sample_data["forwards"][0])

Top-level keys: ['forwards', 'defensemen', 'goalies']
Forwards: 16
Defensemen: 9
Goalies: 2


{'id': 8484153,
 'headshot': 'https://assets.nhle.com/mugs/nhl/20262027/ANA/8484153.png',
 'firstName': {'default': 'Leo'},
 'lastName': {'default': 'Carlsson'},
 'sweaterNumber': 91,
 'positionCode': 'C',
 'shootsCatches': 'L',
 'heightInInches': 75,
 'weightInPounds': 215,
 'heightInCentimeters': 191,
 'weightInKilograms': 98,
 'birthDate': '2004-12-26',
 'birthCity': {'default': 'Karlstad'},
 'birthCountry': 'SWE'}

## Flatten one team's roster

Values are converted to plain Python types here (`None` instead of `NaN`), because MySQL rejects `NaN`
and pandas turns an integer column with a missing value into floats (jersey 12 -> 12.0).

In [7]:
VALID_POSITIONS = {"C", "L", "LW", "R", "RW", "D", "G"}

def text_val(v):
    if isinstance(v, dict):
        v = v.get("default")
    if v is None:
        return None
    v = str(v).strip()
    return v or None

def int_val(v):
    try:
        return int(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def float_val(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def date_val(v):
    return v if isinstance(v, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", v) else None

def flatten_roster(roster_json, team_id):
    """Flatten forwards + defensemen + goalies into one list of player dicts."""
    players = []
    for group in ("forwards", "defensemen", "goalies"):
        for p in roster_json.get(group) or []:
            first, last = text_val(p.get("firstName")), text_val(p.get("lastName"))
            if p.get("id") is None or not first or not last:
                continue   # skip records missing the required fields
            pos = text_val(p.get("positionCode"))
            players.append({
                "player_id": int(p["id"]),
                "team_id": int(team_id),          # our FK, not from the API
                "first_name": first[:100],
                "last_name": last[:100],
                "position": pos if pos in VALID_POSITIONS else None,
                "jersey_number": int_val(p.get("sweaterNumber")),
                "birth_date": date_val(p.get("birthDate")),
                "birth_country": (text_val(p.get("birthCountry")) or "")[:10] or None,
                "height_cm": float_val(p.get("heightInCentimeters")),
                "weight_kg": float_val(p.get("weightInKilograms")),
                "shoots_catches": (text_val(p.get("shootsCatches")) or "")[:5] or None,
                "headshot_url": text_val(p.get("headshot")),
            })
    return players

test_players = flatten_roster(sample_data, team_id_lookup[sample_abbrev])
print(f"Flattened {len(test_players)} players for {sample_abbrev}")
display(pd.DataFrame(test_players).head())

Flattened 27 players for ANA


,player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url
0,8484153,26,Leo,Carlsson,C,91,2004-12-26,SWE,191.0,98.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
1,8481538,26,Judd,Caulfield,R,28,2001-03-19,USA,191.0,100.0,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
2,8482118,26,Sam,Colangelo,R,12,2001-12-26,USA,188.0,95.0,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
3,8483445,26,Cutter,Gauthier,L,61,2004-01-19,SWE,188.0,93.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
4,8475798,26,Mikael,Granlund,C,64,1992-02-26,FIN,178.0,87.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...


## Loop over all teams, fetch + flatten (with one retry pass)

In [8]:
all_players = []
failed_teams = []

def collect(abbrev):
    roster_json = fetch_roster(abbrev)
    rows = flatten_roster(roster_json, team_id_lookup[abbrev])
    all_players.extend(rows)
    return len(rows)

for abbrev in teams_df.team_abbrev:
    try:
        print(f"{abbrev}: OK ({collect(abbrev)} players)")
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"{abbrev}: FAILED - {e}")
        failed_teams.append(abbrev)
    time.sleep(REQUEST_DELAY)

# second chance for failures
still_failed = []
for abbrev in failed_teams:
    time.sleep(2)
    try:
        print(f"{abbrev}: retry OK ({collect(abbrev)} players)")
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"{abbrev}: retry FAILED - {e}")
        still_failed.append(abbrev)

print(f"\nTotal players collected: {len(all_players)}")
if still_failed:
    raise RuntimeError(f"Teams still failing: {still_failed}. Nothing was saved; fix and re-run this cell.")

ANA: OK (27 players)
BOS: OK (23 players)
BUF: OK (27 players)
CAR: OK (24 players)
CBJ: OK (23 players)
CGY: OK (27 players)
CHI: OK (24 players)
COL: OK (23 players)
DAL: OK (24 players)
DET: OK (24 players)
EDM: OK (29 players)
FLA: OK (26 players)
LAK: OK (25 players)
MIN: OK (28 players)
MTL: OK (25 players)
NJD: OK (23 players)
NSH: OK (24 players)
NYI: OK (25 players)
NYR: OK (26 players)
OTT: OK (24 players)
PHI: OK (27 players)
PIT: OK (26 players)
SEA: OK (26 players)
SJS: OK (24 players)
STL: OK (26 players)
TBL: OK (24 players)
TOR: OK (28 players)
UTA: OK (23 players)
VAN: OK (25 players)
VGK: OK (24 players)
WPG: OK (25 players)
WSH: OK (25 players)

Total players collected: 804


## Check for duplicates

In [9]:
unique = {}
for p in all_players:
    unique[p["player_id"]] = p        # same player twice -> keep the last one
players = list(unique.values())
print(f"Dropped {len(all_players) - len(players)} duplicate player_id rows")

players_df = pd.DataFrame(players)
display(players_df.dtypes.to_frame("dtype").T, players_df.head())

Dropped 0 duplicate player_id rows


,player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url
dtype,int64,int64,str,str,str,int64,str,str,float64,float64,str,str


,player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url
0,8484153,26,Leo,Carlsson,C,91,2004-12-26,SWE,191.0,98.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
1,8481538,26,Judd,Caulfield,R,28,2001-03-19,USA,191.0,100.0,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
2,8482118,26,Sam,Colangelo,R,12,2001-12-26,USA,188.0,95.0,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
3,8483445,26,Cutter,Gauthier,L,61,2004-01-19,SWE,188.0,93.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
4,8475798,26,Mikael,Granlund,C,64,1992-02-26,FIN,178.0,87.0,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...


## Load into the `players` table

Uses an **upsert** (`ON DUPLICATE KEY UPDATE`) instead of `INSERT IGNORE`, so a traded player's new team,
jersey number, etc. are updated when you re-run. The whole load is one transaction: all rows or none.

In [10]:
COLUMNS = ["player_id", "team_id", "first_name", "last_name", "position", "jersey_number",
           "birth_date", "birth_country", "height_cm", "weight_kg", "shoots_catches", "headshot_url"]

UPSERT_SQL = f"""
INSERT INTO players ({", ".join(COLUMNS)})
VALUES ({", ".join(["%s"] * len(COLUMNS))})
ON DUPLICATE KEY UPDATE
    {", ".join(f"{c}=VALUES({c})" for c in COLUMNS[1:])}
"""

rows = [tuple(p[c] for c in COLUMNS) for p in players]

try:
    conn.start_transaction()
    cur.executemany(UPSERT_SQL, rows)
    conn.commit()
    cur.execute("SELECT COUNT(*) FROM players")
    print("Rows in players:", cur.fetchone()[0])
except Exception:
    conn.rollback()
    raise

Rows in players: 804


## Sanity checks

In [11]:
cur.execute("SELECT position, COUNT(*) AS player_count FROM players GROUP BY position ORDER BY player_count DESC")
display(pd.DataFrame(cur.fetchall(), columns=["position", "player_count"]))

,position,player_count
0,D,257
1,C,226
2,R,122
3,L,122
4,G,77


In [12]:
cur.execute("""
    SELECT t.team_abbrev, COUNT(p.player_id) AS roster_size
    FROM teams t
    LEFT JOIN players p ON t.team_id = p.team_id
    GROUP BY t.team_abbrev
    ORDER BY roster_size ASC
""")
check = pd.DataFrame(cur.fetchall(), columns=["team_abbrev", "roster_size"])
display(check.head(10))
if (check.roster_size == 0).any():
    print("WARNING: some teams have no players:", list(check[check.roster_size == 0].team_abbrev))

,team_abbrev,roster_size
0,BOS,23
1,CBJ,23
2,COL,23
3,NJD,23
4,UTA,23
5,CAR,24
6,CHI,24
7,DAL,24
8,DET,24
9,NSH,24


In [13]:
cur.close()
conn.close()
session.close()